In [ ]:
# =============================================================================
# PARAMETER REFERENCE — RTX 3060 laptop (6 GB VRAM) + 32 GB RAM
# Model: deepseek-r1:14b (Q4_K_M, ~9 GB). KV cache q8_0 ≈ 96 KB/token.
# RAM ceiling for KV cache before disk swap: ~96k ctx. Stay at or below it.
# =============================================================================
#
# THE FOUR KNOBS
#
#   num_ctx      Context window in tokens. Holds prompt + docs + thinking
#                + final answer. Too small → truncation. Too big → slower
#                prefill and more RAM.
#
#   num_predict  Hard cap on generated tokens (thinking + final answer
#                combined for R1). Cap, not target — model stops when done.
#                Too low → cut-off mid-answer. Too high → no penalty except
#                the theoretical max.
#
#   num_batch    Tokens processed in parallel during prompt ingestion.
#                Does NOT affect generation speed. On 6 GB VRAM, 512 is the
#                ceiling worth using. 256 if you hit OOM. Never raise above
#                512 on this hardware — steals VRAM from model weights.
#
#   temperature  0.6 default. 0.5 for math/algorithmic reasoning (more
#                deterministic). 0.7-0.8 for brainstorming. Leave alone
#                otherwise.
#
# =============================================================================
# QUICK LOOKUP BY TASK (Optimized for RTX 3060 Laptop)
# =============================================================================
#
# TASK                        num_ctx  num_predict  num_batch  temperature
# --------------------------  -------  -----------  ---------  -----------
# Quick chat / fact check        8192         512        512          0.6
# Simple code (< 200 lines)     16384        2048        512          0.6
# Moderate module (~500 lines)  32768        4096        512          0.6
# Complex code / large file     49152        6144        512          0.6
# Single scientific paper       32768        5120        512          0.6
# 2-3 files cross-reference     49152        6144        512          0.6
# 3-4 files, deep analysis      65536        8192        512          0.6
# 5+ files, full review         98304       12288        512          0.6
# Math / derivation / proof     32768        8192        512          0.5
# "Let it cook" deep dive       65536       12288        512          0.5
#
# =============================================================================
# RULES OF THUMB
# =============================================================================
#
# 1. If the answer gets CUT OFF mid-sentence → raise num_predict.
#    For R1, thinking eats budget too. Rough split: ~60% thinking, ~40%
#    answer. So a 4096 cap ≈ 1600 tokens of actual answer.
#
# 2. If the model says "I don't see the full file" or ignores part of a
#    document → raise num_ctx.
#
# 3. If things get SLOWER unexpectedly after a change → you probably pushed
#    num_ctx high enough to spill KV cache to disk. Check `free -h` while
#    running. If swap is being used, drop num_ctx.
#
# 4. Never touch num_batch on this hardware. 512 is the sweet spot.
#
# 5. Switching num_ctx mid-session re-evaluates the ENTIRE history at the
#    new size. Expect a pause on the first call after a switch. If you're
#    doing heavy work, start at the high value and stay there.
#
# 6. Rule of thumb for num_ctx given N tokens of input:
#       num_ctx ≈ input_tokens + num_predict + 512 (safety)
#    Characters ÷ 4 ≈ tokens for English. So a 30,000-char doc ≈ 7,500 tok.
#
# =============================================================================
# EXAMPLE CALLS
# =============================================================================
#
# Quick question:
#   ai.chat(prompt="What is OBIA?",
#           num_ctx=8192, num_predict=512)
#
# Review one large notebook:
#   ai.chat(prompt="Check this for bugs and edge cases.",
#           file_names=["code.ipynb"],
#           num_ctx=49152, num_predict=6144)
#
# Cross-reference three scripts:
#   ai.chat(prompt="Do these agree on the CRS convention?",
#           file_names=["load.py", "reproject.py", "export.py"],
#           num_ctx=65536, num_predict=8192)
#
# Derive something (lower temp):
#   ai.chat(prompt="Derive the gradient of this loss w.r.t. weights.",
#           num_ctx=32768, num_predict=8192, temperature=0.5)
#
# =============================================================================

# Test the basic chat bot

In [1]:
from main import ScientificAssistant

In [2]:
assistant = ScientificAssistant(model_name="gemma4:e4b")
assistant.clear_memory()
assistant.chat(
    prompt="""
    Just say hi. I am checking that you are there.
    """,
    num_ctx=128,
    num_batch=128,
    num_predict=128
)

🧠 Assistant memory cleared.
🤖 ctx=128 temp=0.6 | Tools Active: True



Hello! I'm here and ready to help you with any questions you have. 😊



✅ Turn Complete.
💾 Saved 3 messages and 0 documents to outputs/chat_history/chat_history.json


# Testing gee-mcp integration

In [3]:
import os
from main import ScientificAssistant

os.makedirs("outputs", exist_ok=True)
dummy_script = """
    import ee
    ee.Initialize()

    # Assume standard surface temperature threshold for heat island
    def identify_heat_islands(image):
        lst = image.select('LST_Day_1km')
        # Assumption: Temps > 35C indicate urban heat island. Might need validation.
        return lst.gt(35).rename('heat_island_mask')
"""
with open("outputs/dummy_gee.py", "w") as f:
    f.write(dummy_script)

mcp_env = {
    "GEMINI_API_KEY": "<gemini key>",
    "GEE_PROJECT": "<gee project>",
    "LLM_PROVIDER": "google", #can you openai or other     
    "LLM_NAME": "gemini-3.1-flash-lite" #can you other
}

# Step 1: Initialize the assistant with a tool capable model
assistant = ScientificAssistant(model_name="gemma4:e4b",
    mcp_command="poetry",
    mcp_args=["run", "python", "-m", "gee_mcp.server"],
    env_vars=mcp_env)
# Step 2: Clear memory for testing
assistant.clear_memory()

# Step 3: Initiate chat
prompt = "Load the dummy_gee.py script and use the 'extract_factuality_issues' tool to analyze the scientific assumptions in it."

assistant.chat(
    prompt=prompt,
    file_names=["outputs/dummy_gee.py"]
)

🧠 Assistant memory cleared.
📄 Added 'dummy_gee.py' (322 characters)
🤖 ctx=8192 temp=0.6 | Tools Active: True


⚙️ Triggering MCP Tool: extract_factuality_issues...
⚠️ Attempt 1/3 failed: MCP connection/startup error: MCPError: Connection closed


I encountered a connection error while running the `extract_factuality_issues` tool. However, based on the code provided and the question asked, I can analyze the scientific assumptions for you.

The primary scientific and data assumptions embedded in the `dummy_gee.py` script are:

### 1. Temperature Threshold Assumption
*   **Issue:** The script assumes that a single, fixed surface temperature threshold (specifically **`> 35C`**) is sufficient and scientifically accurate to delineate an "urban heat island" (UHI).
*   **Factuality Concern:** The definition of a UHI is complex and varies geographically and scientifically. A single temperature cutoff might fail to capture the full scope of UHI effects, which are often measured relative to a surrounding rural or baseline area, rather than an absolute value.
*   **Validation Needed:** This threshold needs validation against local climate data, literature reviews, and comparative analysis (e.g., comparing the threshold in different urban environments).

### 2. Data Variable Assumption
*   **Issue:** The script relies exclusively on the `LST_Day_1km` band (Land Surface Temperature).
*   **Factuality Concern:** While LST is a key indicator, it is only one component. The severity and nature of a UHI are also influenced by factors not included, such as:
    *   **Land Cover Type:** (e.g., Impervious surface fraction, vegetation index).
    *   **Atmospheric Conditions:** (e.g., humidity, wind patterns).
    *   **Temporal Variability:** (e.g., time of day/year).

### Summary of Required Verification
To make this analysis scientifically rigorous, the following assumptions must be verified or refined:

1.  **Refining the Definition:** The method should ideally compare the LST of the target area against a reference or control area (e.g., surrounding rural land) rather than using an absolute global threshold.
2.  **Contextualizing the Threshold:** The threshold value (35°C) should be treated as a hypothesis and tested across different geographical regions, as appropriate temperatures vary significantly globally.
3.  **Multivariate Approach:** Incorporating additional indices (like NDVI or impervious surface mapping) alongside LST would significantly improve the scientific robustness of the identification process.



✅ Turn Complete.
💾 Saved 5 messages and 1 documents to outputs/chat_history/chat_history.json
